# DeNoiseRaw を Google Colab で試す

DSLR/ミラーレスのRAWファイルを、センサーノイズの物理モデルに基づいてノイズ除去するツールです。
このノートブックは上から順に実行するだけで、**手持ちのRAWファイルをアップロードしてノイズ除去 → 結果をダウンロード** まで完結します。

- リポジトリ: https://github.com/okahaya/DeNoiseRaw
- 学習済みモデルは同梱していないので、ここでは学習不要の「古典手法」(BM3D / wavelet)を使います。
- GPUは無くても動きます(Runtime → Change runtime type で有効化すれば、将来モデルを学習させる際に使えます)。


## ① インストール

リポジトリを取得して依存関係を入れます。数分かかります。

In [ ]:
# 現時点ではまだ main ブランチに未マージのため、開発ブランチを直接指定しています。
# main にマージされた後は -b オプションを外して構いません。
!git clone -q -b claude/dslr-noise-reduction-app-5rawrf https://github.com/okahaya/DeNoiseRaw.git
%cd DeNoiseRaw
!pip install -q -e ".[torch,gui]"

import denoiseraw
print("denoiseraw", denoiseraw.__version__, "を読み込みました")


## ② RAWファイルをアップロード

「ファイル選択」ダイアログが出るので、手持ちのRAWファイル(CR2/CR3/NEF/ARW/RAF/RW2/DNG等)を選んでください。複数選択も可能です。


In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    raw_paths = list(uploaded.keys())
else:
    # Colab以外(ローカルやテスト実行)では、このリストを手動で編集してください。
    raw_paths = []
    print("Colab環境ではないため、raw_paths に直接ファイルパスを入れて次に進んでください。")

print(f"{len(raw_paths)} 件のファイルを受け取りました:", raw_paths)


## ③ ノイズ除去を実行

In [ ]:
from denoiseraw import DenoiseSettings, denoise_file, write_outputs

# strength: 1.0が基準。小さいほど粒状感を残し、大きいほど強く均す。
# classical_backend: "auto"(bm3dが入っていれば高品質・低速、無ければ高速なwavelet)。
settings = DenoiseSettings(method="classical", classical_backend="auto", strength=1.0)

results = []
for path in raw_paths:
    result = denoise_file(path, settings)
    out_path = path.rsplit(".", 1)[0] + "_denoised.dng"
    write_outputs(result, out_path, settings)
    m = result.metrics
    print(f"{path}: {result.elapsed:.1f}秒  "
          f"ノイズ {m['residual_noise_before']:.5f} -> {m['residual_noise_after']:.5f} "
          f"({m.get('noise_reduction_db', 0):.1f} dB)  -> {out_path}")
    results.append((path, out_path, result))


## ④ 処理前後を見比べる

In [ ]:
import matplotlib.pyplot as plt
from denoiseraw.rawio.loader import load_raw
from denoiseraw.rawio.develop import develop

if results:
    path, out_path, result = results[0]
    before = develop(load_raw(path), auto_bright=True)
    after = result.preview()

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    # matplotlibの標準フォントはCJKグリフを持たないため、図中ラベルは英語にしています。
    axes[0].imshow(before); axes[0].set_title("Before"); axes[0].axis("off")
    axes[1].imshow(after); axes[1].set_title("After"); axes[1].axis("off")
    fig.tight_layout()
    plt.show()
else:
    print("先に②③を実行してください。")


## ⑤ 結果をダウンロード

In [ ]:
if IN_COLAB:
    for _, out_path, _ in results:
        files.download(out_path)
else:
    print("出力ファイル:", [out_path for _, out_path, _ in results])


## (おまけ) ポチポチ操作できるGUIをスマホからも開く

下のセルを実行すると、複数ファイル選択・設定・実行ボタン付きのブラウザGUIが起動し、
`https://xxxxx.gradio.live` という一時的な公開リンクが出力されます。
このリンクはスマホのブラウザからもそのまま開けます(数時間で失効します)。

止めたいときはセルを中断(■ボタン)してください。


In [ ]:
!denoiseraw gui --share
